**Population for Tdb (Integrated malaria dashboard indicator DB tables)**  

For this update we take the latest available population and pyramid (OU) data downloaded from SNIS.  
pipeline responsable : *DHIS2 SNIS Analytics Extract*

> At the start of the year, there will be no population available, therefore we use the previous year data..  
> Once 90% (completeness) of the data has been uploaded for the current year (ANNEE_A_ANALYSER) we can start using the new population data.

In [ ]:
# !pip install openhexa.toolbox==2.10.0
# !pip show openhexa.toolbox

In [ ]:
from openhexa.sdk import workspace, current_run
from openhexa.toolbox.dhis2 import DHIS2, periods
from openhexa.toolbox.dhis2.dataframe import InvalidParameterError, get_organisation_unit_levels
# from openhexa.toolbox.dhis2.dataframe import get_organisation_units
import os
from pathlib import Path
import pandas as pd
import polars as pl
import numpy as np

from datetime import datetime
from openhexa.toolbox.dhis2 import DHIS2, periods

Retrieve population data FROM: 2017 UNTIL: "ANNEE_A_ANALYSER", and create population file for tdb: pnlp-tdb-pipeline/data/raw/population/analytics.csv.  
The population is only used for yearly indicators such as incidence and mortality (see: pnlp-tdb-pipeline/03a_indicator_computation_extra_indicators.ipynb).  
  
(!)In the indicators calculation (03a_indicators_computation_extra_indicators) we select the population matching the ANNEE_A_ANALYSER.  
**So make sure we have data for that year** (e.g; ANNEE_A_ANALYSER=2025 --> the population file should contain data for that year ofc).


>NOTE: Run this script once a year! and manually CHECK if we have full data available (for all aires de santé).  
We have population data completed for year 2024. Once there's population data for 2025, we can run this  
code and update the population file (around March or April of the current year most probably). THIS SHOULD HAPPEN AUTOMATICALLY HERE

In [ ]:
# Parameter 
# ANNEE_A_ANALYSER = 2026

In [ ]:
# completeness tolerance (if above we use the new data, otherwise we use previous year data)
tolerance = 90

In [ ]:
# column formats
dtypes_dict = {
    'dx_uid': np.dtype('O'),
    'period': np.dtype('int64'),
    'ou_id': np.dtype('O'),
    'coc_id': np.dtype('O'),
    'value': np.dtype('int64'),
    'dx_name': np.dtype('O'),
    'coc_name': np.dtype('O'),
    'ou_name': np.dtype('O'),
    'level_1_uid': np.dtype('O'),
    'level_1_name': np.dtype('O'),
    'level_2_uid': np.dtype('O'),
    'level_2_name': np.dtype('O'),
    'level_3_uid': np.dtype('O'),
    'level_3_name': np.dtype('O'),
    'level_4_uid': np.dtype('O'),
    'level_4_name': np.dtype('O')
}

# helper functions
def is_data_available(pop_extract: pd.DataFrame, required_ou: pd.DataFrame, tolerance:int=90) -> bool:
    if pop_extract.empty:
        print(f"No data available!.")
        return False
    
    merged_df = required_ou.merge(pop_extract, left_on="id", right_on="org_unit", how="left")
    total_rows = len(merged_df)
    merged_df["value"] = pd.to_numeric(merged_df["value"], errors="coerce")
    na_count = merged_df["value"].isna().sum()
    zero_count = (merged_df["value"] == 0).sum()
    completion = ((total_rows - (na_count + zero_count)) / total_rows) * 100
    print(f"Completed datapoints: {completion:.2f}%")
    return bool(completion>=tolerance)


# format output pop table
def perform_formatting(hist_population: pd.DataFrame, new_population: pd.DataFrame, org_units: pd.DataFrame, dtypes_dict: dict) -> pd.DataFrame:
    
    # Merge metadata and format
    merged = new_population.merge(org_units, left_on="org_unit", right_on="id" , how="left")
    merged = merged.drop(columns=["data_type", "attribute_option_combo", "rate_metric", "domain_type", 
                                  "id", "level", "opening_date", "closed_date", "geometry", "level_5_id",
                                  "level_5_name", "level_6_id", "level_6_name"],
                         errors="ignore")
    
    # retrieve and merge missing metadata
    df = dhis2_source.meta.add_dx_name_column(dataframe=merged, dx_id_column="dx")
    df = dhis2_source.meta.add_coc_name_column(dataframe=df, coc_column="category_option_combo")
    df.rename(columns={"dx":"dx_uid", "org_unit":"ou", "name":"ou_name", "category_option_combo":"coc_id"}, inplace=True)

    # Rename
    df_f = df.rename(columns={"ou":"ou_id", "co_name":"coc_name"} )
    parent_names = {
        'level_1_id': 'level_1_uid', 'level_1_name':'level_1_name',
        'level_2_id': 'level_2_uid', 'level_2_name':'level_2_name',
        'level_3_id': 'level_3_uid', 'level_3_name': 'level_3_name',
        'level_4_id': 'level_4_uid', 'level_4_name': 'level_4_name'
    }
    df_f = df_f.rename(columns=parent_names)

    # concat (use same column types)
    df_f_s = df_f[hist_population.columns].astype(dtypes_dict)
    final_df = pd.concat([hist_population, df_f_s], ignore_index=True)  
    final_df = final_df.sort_values(by='period', ascending=True)

    return final_df



def get_organisation_units(
    dhis2: DHIS2, max_level: int | None = None, filters: list[str] | None = None
) -> pl.DataFrame:
    """Extract organisation units metadata.

    Parameters
    ----------
    dhis2 : DHIS2
        DHIS2 instance.
    max_level : int, optional
        Maximum level of organisation units to extract. If None, all levels are extracted.
    filters : list[str], optional
        DHIS2 query filter expressions.

    Returns
    -------
    pl.DataFrame
        Dataframe containing organisation units metadata with the following columns: id, name,
        level, level_{level}_id, level_{level}_name, geometry.

    Raises
    ------
    InvalidParameter
        If max_level is greater than the maximum level of the organisation units.
    """
    levels = get_organisation_unit_levels(dhis2)
    if max_level:
        if max_level > levels["level"].max():
            msg = f"max_level cannot be greater than {levels['level'].max()}"
            current_run.log_error(msg)
            raise InvalidParameterError(msg)
        level_filter = f"level:le:{max_level}"
        if filters:
            filters = [*filters, max_level]
        else:
            filters = [level_filter]
    
    meta = dhis2.meta.organisation_units()

    schema = {
        "id": str,
        "name": str,
        "level": int,
        "path": str,
        "openingDate": str,
        "geometry": str,
    }
    df = pl.DataFrame(data=meta, schema=schema)

    for row in levels.iter_rows(named=True):
        lvl = row["level"]
        if max_level:
            if lvl > max_level:
                continue

        df = df.with_columns(
            pl.col("path").str.split("/").list.slice(1).list.get(lvl - 1, null_on_oob=True).alias(f"level_{lvl}_id")
        )

        df = df.join(
            other=df.select("id", pl.col("name").alias(f"level_{lvl}_name")),
            left_on=f"level_{lvl}_id",
            right_on="id",
            how="left",
        )

    df = df.select(
        "id",
        "name",
        "level",
        pl.col("openingDate").str.to_datetime("%Y-%m-%dT%H:%M:%S.%3f").alias("opening_date"),
        *[col for col in df.columns if col.startswith("level_")],
        "geometry",
    )

    return df.sort(by=["level", "name"], descending=False)


**Connect to dhis2 and load ous**  

In [ ]:
# establish DHIS2 connection
connection = workspace.dhis2_connection("drc-snis")
dhis2_source = DHIS2(connection=connection, cache_dir=None)  # f'{workspace.files_path}/temp/')

**Load historic population data (Aire de sante)**

In [ ]:
# Load population extract (SNIS Analytics extract pipeline /data/raw/population)
current_pop_data = pd.read_csv(f"/home/jovyan/workspace/pnlp-tdb-pipeline/data/raw/population/analytics_{ANNEE_A_ANALYSER - 1}.csv")     # Last file available (historic)

print(f"Periods: {current_pop_data.period.unique()}")
current_pop_data.head(2)

**Load population data (Aire de sante) from SNIS Extracts**

In [ ]:
# new_extract_pop = pd.read_parquet('/home/jovyan/workspace/pipelines/dhis2_snis_extract/data/raw/population/snis_population.parquet') # Raw dhis2 data for year to update (level 4)

# SNIS dhis2 data for year to update (level 4) data download from DATASET.. look at pipeline code (python : drc-pnlp-tdb)
try:
    new_extract_pop = pd.read_parquet(f'/home/jovyan/workspace/pnlp-tdb-pipeline/data/snis_population/snis_population_{ANNEE_A_ANALYSER}.parquet')
except Exception: 
    current_run.log_error(f"Requested population data for year {ANNEE_A_ANALYSER} is not available.")
    raise

# quick format
new_extract_pop["period"] = pd.to_numeric(new_extract_pop["period"], errors="coerce")
new_extract_pop["value"] = pd.to_numeric(new_extract_pop["value"], errors="coerce")

print(f"Period: {new_extract_pop.period.unique()}")
print(f"Total pop: {new_extract_pop.value.sum()}")
new_extract_pop.head(2)

**Load pyramid (org units) from SNIS Extracts**

In [ ]:
# org_units_extract = pd.read_parquet(f"/home/jovyan/workspace/pnlp-tdb-pipeline/data/snis_pyramid/snis_pyramid.parquet")     # Last file available from pipeline

In [ ]:
# get pyramid
org_units = get_organisation_units(dhis2_source)
org_units_pd = org_units.to_pandas()
org_units_pd = org_units_pd[org_units_pd.level == 4]
org_units_pd = org_units_pd.drop(columns=['opening_date', 'closed_date', 'parent', 'level', 'geometry', 'level_5_id', 'level_5_name'], errors="ignore")    

print(f"Total OUs at level 4: {org_units_pd.shape}")
org_units_pd.head()

**Check if there's available (complete) data for `ANNEE_A_ANALYSER`**

In [ ]:
# check if data is available
data_available = is_data_available(
    pop_extract=new_extract_pop[new_extract_pop.period == int(ANNEE_A_ANALYSER)], 
    required_ou=org_units_pd,
    tolerance=tolerance)

print(f"Data available: {data_available}")

In [ ]:

# If we have complete data (over tolerance threshold), we can overwrite the population file "population/analytics.csv"
if data_available:
    year_selection = int(ANNEE_A_ANALYSER)
    if current_run:
        current_run.log_info(f"Data for year {year_selection} is complete.")
    print(f"Data for year {year_selection} is complete.")
    pop_data = current_pop_data[current_pop_data["period"] != year_selection]
    extract_pop = new_extract_pop[new_extract_pop["period"] == year_selection]
    print(f"Selecting population from historic data: {pop_data.period.unique()}")    
    if current_run:
        current_run.log_info(f"Selecting new year population from extract: {extract_pop.period.unique()}")
    print(f"Selecting new year population from extract: {extract_pop.period.unique()}")
    final_table = perform_formatting(
        hist_population=pop_data, 
        new_population=extract_pop, 
        org_units=org_units_pd, 
        dtypes_dict=dtypes_dict)
    
else:
    year_selection = int(ANNEE_A_ANALYSER) - 1
    if current_run:
        current_run.log_info(f"Data for year {ANNEE_A_ANALYSER} not complete, falling back to {year_selection} data (copy).")
    print(f"Data for year {ANNEE_A_ANALYSER} not complete, falling back to {year_selection} data (copy).")    
    last_extract_pop = pd.read_csv(f"/home/jovyan/workspace/pnlp-tdb-pipeline/data/raw/population/analytics_{year_selection}.csv")
    pop_prev_year = last_extract_pop[last_extract_pop["period"] == year_selection].copy()
    pop_prev_year["period"] = int(ANNEE_A_ANALYSER)
    final_table = pd.concat([last_extract_pop, pop_prev_year], ignore_index=True)
    final_table = final_table.astype(dtypes_dict)


In [ ]:
# Print summary ..
periods_available = final_table.period.unique()
print(f"Periods available: {periods_available}")
for p in periods_available:
    print(f"Period {p} aires de santé : {final_table[final_table.period==p].level_4_uid.count()} total pop: {final_table[final_table.period==p].value.apply(lambda x: float(x)).sum()}")

**Save the population data for TDB for year**

In [ ]:
if not final_table.empty:
    final_table.to_csv("/home/jovyan/workspace/pnlp-tdb-pipeline/data/raw/population/analytics.csv", index=False)
    final_table.to_csv(f"/home/jovyan/workspace/pnlp-tdb-pipeline/data/raw/population/analytics_{int(ANNEE_A_ANALYSER)}.csv", index=False)

if current_run:
    current_run.log_info(f"Pop data saved: /pnlp-tdb-pipeline/data/raw/population/analytics_{int(ANNEE_A_ANALYSER)}.csv")